Structured output is a technique in LangChain that allows an LLM to return responses in a predefined, predictable format instead of unstructured text. We can define a schema specifying the required fields and their data types, and the model formats its response according to that schema. LangChain provides methods such as with_structured_output() to connect the model with schemas defined using Pydantic, TypedDict, or JSON Schema. Structured output is useful when the model's response needs to be processed programmatically, such as extracting user information, classifying text, generating API-ready data, or building reliable AI applications.

### Pydantic

**Pydantic** is a Python library used for **data validation and structured data modeling** using Python type annotations. In LangChain, Pydantic models are commonly used to define the expected structure of an LLM's output, including the field names, data types, and validation rules. When used with `with_structured_output()`, the model generates data according to the defined Pydantic schema, making the output predictable, validated, and easy to use programmatically.


In [4]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from pydantic import BaseModel, Field

load_dotenv()

model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

class Series(BaseModel):
    name: str = Field(description="Name of the series")
    genre: str = Field(description="Genre of the series")
    release_year: int = Field(description="Release year")
    rating: float = Field(description="Rating of the series")
    cast: list[str] = Field(description="Main cast members")

structured_model = model.with_structured_output(
    Series,
    method="json_schema"
)

response = structured_model.invoke(
    "Give me information about the series Stranger Things."
)

print(response)
print(response.name)
print(response.genre)
print(response.release_year)
print(response.rating)
print(response.cast)

name='Stranger Things' genre='Science Fiction, Horror' release_year=2016 rating=8.7 cast=['Winona Ryder', 'David Harbour', 'Finn Wolfhard', 'Millie Bobby Brown', 'Gaten Matarazzo']
Stranger Things
Science Fiction, Horror
2016
8.7
['Winona Ryder', 'David Harbour', 'Finn Wolfhard', 'Millie Bobby Brown', 'Gaten Matarazzo']


In [3]:
import langchain
import langchain_core
import langchain_groq
import pydantic

print("langchain:", langchain.__version__)
print("langchain-core:", langchain_core.__version__)
print("langchain-groq:", langchain_groq.__version__)
print("pydantic:", pydantic.__version__)

langchain: 1.3.14
langchain-core: 1.5.3
langchain-groq: 1.1.3
pydantic: 2.13.4


In [5]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from pydantic import BaseModel, Field

load_dotenv()

model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

class Series(BaseModel):
    name: str = Field(description="Name of the series")
    genre: str = Field(description="Genre of the series")
    release_year: int = Field(description="Release year")
    rating: float = Field(description="Rating of the series")
    cast: list[str] = Field(description="Main cast members")

structured_model = model.with_structured_output(
    Series,
    method="json_schema",
    include_raw=True
)

response = structured_model.invoke(
    "Give me information about the series Stranger Things."
)

print("RAW:")
print(response["raw"])

print("\nPARSED:")
print(response["parsed"])

print("\nPARSING ERROR:")
print(response["parsing_error"])

RAW:
content='{"name":"Stranger Things","genre":"Science Fiction, Horror","release_year":2016,"rating":8.7,"cast":["Winona Ryder","David Harbour","Finn Wolfhard","Millie Bobby Brown","Gaten Matarazzo"]}' additional_kwargs={'reasoning_content': 'We need to output JSON object following the schema. Must include all required fields: name, genre, release_year, rating, cast. Provide values for Stranger Things. Use compact JSON formatting. Ensure valid JSON. Provide rating maybe 8.7? Let\'s choose rating 8.7. Release year 2016. Cast: "Winona Ryder", "David Harbour", "Finn Wolfhard", "Millie Bobby Brown", "Gaten Matarazzo". Genre: "Science Fiction, Horror". Provide description? Not required but can include optional. But schema only requires those fields. We can include description but not required. But we can include it. It\'s okay. Provide final JSON. Ensure no trailing commas. Use compact formatting: no spaces? But can include minimal spaces. Let\'s produce:\n\n{"name":"Stranger Things","gen

In [6]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from pydantic import BaseModel, Field

load_dotenv()

model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

class Actor(BaseModel):
    name: str = Field(description="Actor's name")
    role: str = Field(description="Character played by the actor")


class Movie(BaseModel):
    title: str = Field(description="Movie title")
    genre: str = Field(description="Movie genre")
    release_year: int = Field(description="Release year")
    rating: float = Field(description="Movie rating")
    director: str = Field(description="Director's name")
    cast: list[Actor] = Field(description="List of actors and their roles")


structured_model = model.with_structured_output(
    Movie,
    method="json_schema",
    include_raw=True
)

response = structured_model.invoke(
    "Give me structured information about the movie Interstellar."
)

print("RAW:")
print(response["raw"])

print("\nPARSED:")
print(response["parsed"])

print("\nPARSING ERROR:")
print(response["parsing_error"])

RAW:
content='{"title":"Interstellar","genre":"Science Fiction","release_year":2014,"rating":8.6,"director":"Christopher Nolan","cast":[{"name":"Matthew McConaughey","role":"Cooper"},{"name":"Anne Hathaway","role":"Amelia Brand"},{"name":"Jessica Chastain","role":"Murph (adult)"},{"name":"Michael Caine","role":"Professor Brand"},{"name":"Casey Affleck","role":"Tom Cooper"},{"name":"Mackenzie Foy","role":"Murph (child)"},{"name":"John Lithgow","role":"Dr. Mann"}]}' additional_kwargs={'reasoning_content': 'We need to output JSON in the specified schema. Must be compact JSON. Must include all required fields: title, genre, release_year, rating, director, cast. Provide cast list with name and role. Provide rating number. For Interstellar: title "Interstellar", genre "Science Fiction", release_year 2014, rating maybe 8.6 (IMDb). Director "Christopher Nolan". Cast: Matthew McConaughey as Cooper, Anne Hathaway as Amelia Brand, Jessica Chastain as Murph (adult), Michael Caine as Professor Bran

### TypedDict

**`TypedDict`** is a Python typing feature used to define the expected **structure of a dictionary**, including its keys and corresponding value types. In LangChain, `TypedDict` can be used with structured output to specify what fields the LLM should return without creating a full Pydantic model. Unlike Pydantic, `TypedDict` mainly provides **static type information** and does not perform runtime data validation by itself. It is useful when you want a lightweight schema for structured LLM responses with clearly defined field names and data types.


In [7]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from typing import TypedDict, Annotated

load_dotenv()

model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

class Movie(TypedDict):
    title: Annotated[str, "Name of the movie"]
    genre: Annotated[str, "Genre of the movie"]
    release_year: Annotated[int, "Year the movie was released"]
    rating: Annotated[float, "Movie rating out of 10"]
    director: Annotated[str, "Name of the director"]
    cast: Annotated[list[str], "Names of the main cast members"]

structured_model = model.with_structured_output(
    Movie,
    method="json_schema",
    include_raw=True
)

response = structured_model.invoke(
    "Give me structured information about the movie avengers."
)

print("RAW:")
print(response["raw"])

print("\nPARSED:")
print(response["parsed"])

print("\nPARSING ERROR:")
print(response["parsing_error"])

RAW:
content='{"title":"The Avengers","release_year":2008,"genre":"Action, Adventure, Sci-Fi","director":"Joss Whedon","rating":8.0,"cast":["Robert Downey Jr.","Chris Evans","Mark Ruffalo","Chris Hemsworth","Scarlett Johansson","Jeremy Renner","Tom Hiddleston","Samuel L. Jackson"]}' additional_kwargs={'reasoning_content': 'We need to output JSON that matches the schema. The schema: type object, properties: title, release_year, genre, director, rating, cast. All required? The schema doesn\'t specify required, but we should include all fields. Provide structured info about the movie Avengers. Which Avengers? Likely "The Avengers" (2008). Provide title "The Avengers", release_year 2008, genre "Action, Adventure, Sci-Fi", director "Joss Whedon", rating maybe 8.0? Cast: list of main cast: Robert Downey Jr., Chris Evans, Mark Ruffalo, Chris Hemsworth, Scarlett Johansson, Jeremy Renner, Tom Hiddleston, Samuel L. Jackson, etc. Provide rating out of 10. Use number. Provide cast array of strings

In [10]:
model.profile

{'name': 'GPT OSS 20B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

### Dataclasses

**Dataclasses** are a Python feature used to create classes primarily for storing structured data with less boilerplate code. By using the `@dataclass` decorator, Python automatically provides methods such as `__init__()` and `__repr__()`, while type annotations define the expected data types of fields. In LangChain, dataclasses can be used to represent structured information and define the expected shape of data returned by an LLM, making the output easier to organize and work with programmatically.


In [11]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from pydantic import BaseModel, Field
from dataclasses import dataclass

load_dotenv()

model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)


@dataclass
class Movie:
    title: str
    genre: str
    release_year: int
    rating: float
    director: str


class MovieSchema(BaseModel):
    movie: Movie = Field(description="Structured information about the movie")


structured_model = model.with_structured_output(
    MovieSchema,
    method="json_schema",
    include_raw=True
)

response = structured_model.invoke(
    "Give me structured information about the movie Interstellar."
)

print("RAW:")
print(response["raw"])

print("\nPARSED:")
print(response["parsed"])

print("\nPARSING ERROR:")
print(response["parsing_error"])

RAW:
content='{"movie":{"title":"Interstellar","genre":"Science Fiction","release_year":2014,"rating":8.6,"director":"Christopher Nolan"}}' additional_kwargs={'reasoning_content': 'We need to output JSON that matches the schema. The schema: top-level object with required field "movie". Inside movie: required fields title, genre, release_year, rating, director. Provide values. Must be compact JSON. Ensure all fields present. Provide rating numeric. For Interstellar: title "Interstellar", genre "Science Fiction", release_year 2014, rating maybe 8.6 (IMDb). director "Christopher Nolan". Provide rating as number. Use compact JSON: no spaces. Ensure valid JSON. Let\'s produce.'} response_metadata={'token_usage': {'completion_tokens': 152, 'prompt_tokens': 226, 'total_tokens': 378, 'completion_time': 0.157345921, 'completion_tokens_details': {'reasoning_tokens': 109}, 'prompt_time': 0.012982392, 'prompt_tokens_details': None, 'queue_time': 0.052706767, 'total_time': 0.170328313}, 'model_name

In [12]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from pydantic import BaseModel, Field
from typing import TypedDict, Annotated
from dataclasses import dataclass, asdict

load_dotenv()

model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)


@dataclass
class ContactInfo:
    name: str
    email: str
    phone: str
    company: str


class ContactSchema(TypedDict):
    contact: Annotated[
        ContactInfo,
        "Extract the contact information from the text"
    ]


structured_model = model.with_structured_output(
    ContactSchema,
    method="json_schema"
)

text = """
My name is Rahul Sharma. I work at Infosys.
My email is rahul.sharma@gmail.com and my phone number is
+91 9876543210.
"""

response = structured_model.invoke(text)

print(response)

{'contact': {'name': 'Rahul Sharma', 'email': 'rahul.sharma@gmail.com', 'phone': '+91 9876543210', 'company': 'Infosys'}}


In [14]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from typing import TypedDict
from dataclasses import dataclass, asdict

load_dotenv()

model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)


class ContactSchema(TypedDict):
    name: str
    email: str
    phone: str
    company: str


@dataclass
class ContactInfo:
    name: str
    email: str
    phone: str
    company: str


structured_model = model.with_structured_output(
    ContactSchema,
    method="json_schema"
)

text = """
My name is Rahul Sharma. I work at Infosys.
You can contact me at rahul.sharma@gmail.com
or call me at +91 9876543210.
"""

response = structured_model.invoke(text)

contact = ContactInfo(**response)

contact_dict = asdict(contact)

print("Dataclass:")
print(contact)

print("\nDictionary:")
print(contact_dict)

print("\nName:", contact.name)
print("Email:", contact.email)
print("Phone:", contact.phone)
print("Company:", contact.company)

Dataclass:
ContactInfo(name='Rahul Sharma', email='rahul.sharma@gmail.com', phone='+91 9876543210', company='Infosys')

Dictionary:
{'name': 'Rahul Sharma', 'email': 'rahul.sharma@gmail.com', 'phone': '+91 9876543210', 'company': 'Infosys'}

Name: Rahul Sharma
Email: rahul.sharma@gmail.com
Phone: +91 9876543210
Company: Infosys
